In [93]:
#Importing the libraries
import tkinter as tk
from tkinter import messagebox
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import heapq
import math
import csv

In [ ]:
#Initialising tkinter
root = tk.Tk()
root.title("Flight Route Planner with A* Search")
root.geometry("700x600")

In [98]:
# Constants
FUEL_CONSUMPTION_RATE = 0.05
AVERAGE_SPEED = 850  # speed is in km/hr

In [100]:
coordinates = {}
with open('airports.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        coordinates[row['Airport']] = (float(row['Latitude']), float(row['Longitude']))

# Load the distances between airports from CSV
edges = []
with open('distances.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        start = row['Start_Airport']
        end = row['End_Airport']
        dist = float(row['Distance'])
        edges.append((start, end, dist))

# Build the graph
graph = {}
for start, end, dist in edges:
    graph.setdefault(start, []).append((end, dist))
    graph.setdefault(end, []).append((start, dist))

In [102]:
# Heuristic: Haversine formula
def haversine(coord1, coord2):
    R = 6371  # Radius of the Earth in km
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [104]:
# A* search implementation
def astar(start, goal, factor="distance"):
    open_set = []
    heapq.heappush(open_set, (0, 0, start, []))  # (f, g, current, path)
    visited = set()

    while open_set:
        f, g, current, path = heapq.heappop(open_set)
        if current in visited:
            continue
        visited.add(current)
        path = path + [current]

        if current == goal:
            return path, g

        for neighbor, dist in graph.get(current, []):
            if neighbor in visited:
                continue

            if factor == "distance":
                cost = dist
            elif factor == "fuel":
                cost = dist * FUEL_CONSUMPTION_RATE
            elif factor == "time":
                cost = dist / AVERAGE_SPEED

            g_new = g + cost
            h = haversine(coordinates[neighbor], coordinates[goal])  # Heuristic
            heapq.heappush(open_set, (g_new + h, g_new, neighbor, path))

    return None, None

In [106]:
# GUI Components
tk.Label(root, text="Start Airport:").pack()
start_entry = tk.Entry(root)
start_entry.pack()

tk.Label(root, text="End Airport:").pack()
end_entry = tk.Entry(root)
end_entry.pack()

tk.Label(root, text="Optimization Factor:").pack()
factor_var = tk.StringVar(value="distance")
tk.Radiobutton(root, text="Distance", variable=factor_var, value="distance").pack()
tk.Radiobutton(root, text="Fuel", variable=factor_var, value="fuel").pack()
tk.Radiobutton(root, text="Time", variable=factor_var, value="time").pack()

tk.Label(root, text="Available Airports:\n" + ", ".join(coordinates.keys()), fg="blue").pack(pady=5)

result_text = tk.StringVar()
tk.Label(root, textvariable=result_text, wraplength=600, fg="green").pack(pady=10)

graph_frame = tk.Frame(root)
graph_frame.pack()

In [108]:
# Plot the route
def plot_path(path):
    for widget in graph_frame.winfo_children():
        widget.destroy()

    fig, ax = plt.subplots(figsize=(6, 4))
    pos = {city: (coordinates[city][1], coordinates[city][0]) for city in coordinates}

    # Plot nodes
    for city, (x, y) in pos.items():
        ax.plot(x, y, 'bo')
        ax.text(x + 0.3, y + 0.3, city, fontsize=9)

    # Plot edges
    for start, end, _ in edges:
        x_vals = [pos[start][0], pos[end][0]]
        y_vals = [pos[start][1], pos[end][1]]
        ax.plot(x_vals, y_vals, 'gray', linewidth=1)

    # Plot path
    for i in range(len(path) - 1):
        x_vals = [pos[path[i]][0], pos[path[i + 1]][0]]
        y_vals = [pos[path[i]][1], pos[path[i + 1]][1]]
        ax.plot(x_vals, y_vals, 'red', linewidth=2)

    ax.set_title("Optimal Flight Path")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True)

    canvas = FigureCanvasTkAgg(fig, master=graph_frame)
    canvas.draw()
    canvas.get_tk_widget().pack()

In [110]:
# Handle button click
def find_route():
    start = start_entry.get().strip().upper()
    end = end_entry.get().strip().upper()
    factor = factor_var.get()

    if start not in coordinates or end not in coordinates:
        messagebox.showerror("Error", "Invalid airport code.")
        return

    path, cost = astar(start, end, factor)
    if path:
        unit = {"distance": "km", "fuel": "liters", "time": "hours"}[factor]
        result_text.set(f"Optimal Path: {' → '.join(path)}\nTotal {factor}: {cost:.2f} {unit}")
        plot_path(path)
    else:
        result_text.set("No path found.")

def clear_inputs():
    start_entry.delete(0, tk.END)
    end_entry.delete(0, tk.END)
    result_text.set("")
    for widget in graph_frame.winfo_children():
        widget.destroy()

tk.Button(root, text="Find Route", command=find_route).pack(pady=5)
tk.Button(root, text="Clear", command=clear_inputs).pack(pady=5)


In [ ]:
# Run app
root.mainloop()